<a href="https://colab.research.google.com/github/hariteja273/sentimental-analysis-across-regions/blob/main/sentimental_analysis_across_region.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# INDIA SENTIMENT DASHBOARD (NO WIDGETS)
# ==========================================

!pip install pandas textblob plotly -q

import pandas as pd
from textblob import TextBlob
import plotly.express as px
from collections import Counter

# ------------------------------------------
# STEP 1: Upload CSV
# ------------------------------------------
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("✅ Dataset Loaded")
print(df.head())

# ------------------------------------------
# STEP 2: AUTO DETECT REGION/CITY COLUMN
# ------------------------------------------
region_col = None
review_col = None

for col in df.columns:
    if "city" in col.lower() or "region" in col.lower():
        region_col = col

    if "review" in col.lower():
        review_col = col

if region_col is None or review_col is None:
    raise Exception("❌ CSV must contain review and city/region columns")

df = df[[review_col, region_col]].dropna()
df.columns = ["review", "region"]

# ------------------------------------------
# STEP 3: REGION COORDINATES
# ------------------------------------------
region_coords = {
    "Mumbai": (19.0760, 72.8777),
    "Delhi": (28.7041, 77.1025),
    "Hyderabad": (17.3850, 78.4867),
    "Chennai": (13.0827, 80.2707),
    "Bangalore": (12.9716, 77.5946),
    "Kolkata": (22.5726, 88.3639),
    "Pune": (18.5204, 73.8567),
    "Ahmedabad": (23.0225, 72.5714)
}

# Keep supported regions only
df = df[df['region'].isin(region_coords.keys())]

# Add coordinates
df['lat'] = df['region'].map(lambda x: region_coords[x][0])
df['lon'] = df['region'].map(lambda x: region_coords[x][1])

# ------------------------------------------
# STEP 4: SENTIMENT ANALYSIS
# ------------------------------------------
def get_sentiment(text):

    score = TextBlob(str(text)).sentiment.polarity

    if score > 0:
        return "Positive", score

    elif score < 0:
        return "Negative", score

    else:
        return "Neutral", score


df[['Sentiment', 'Score']] = df['review'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

print("\n✅ Sentiment Analysis Completed")

# ------------------------------------------
# STEP 5: AGGREGATION
# ------------------------------------------
region_avg = df.groupby('region')['Score'].mean().reset_index()

region_counts = df.groupby(
    ['region', 'Sentiment']
).size().unstack().fillna(0).reset_index()

final_df = pd.merge(
    region_avg,
    region_counts,
    on="region"
)

final_df['lat'] = final_df['region'].map(
    lambda x: region_coords[x][0]
)

final_df['lon'] = final_df['region'].map(
    lambda x: region_coords[x][1]
)

print("\n✅ Aggregation Completed")
print(final_df)

# ------------------------------------------
# STEP 6: INDIA SENTIMENT MAP
# ------------------------------------------
fig = px.scatter_geo(
    final_df,
    lat="lat",
    lon="lon",
    color="Score",
    size="Positive",
    hover_name="region",
    hover_data=["Score", "Positive", "Negative", "Neutral"],
    title="India Sentiment Dashboard"
)

fig.update_geos(
    scope="asia",
    center={"lat": 20, "lon": 78},
    projection_scale=4
)

fig.show()

# ------------------------------------------
# STEP 7: REGION SEARCH (TEXT INPUT)
# ------------------------------------------
while True:

    print("\nAvailable Regions:")
    print(final_df['region'].tolist())

    region = input(
        "\nEnter region name (or type exit): "
    )

    if region.lower() == "exit":
        print("Program ended.")
        break

    if region not in final_df['region'].values:
        print("❌ Region not found")
        continue

    data = final_df[
        final_df['region'] == region
    ]

    print("\n======================")
    print("📍 Region:", region)
    print("======================")

    print(
        "⭐ Avg Score:",
        round(data['Score'].values[0], 2)
    )

    print(
        "😊 Positive:",
        int(data.get('Positive', 0))
    )

    print(
        "😐 Neutral:",
        int(data.get('Neutral', 0))
    )

    print(
        "😡 Negative:",
        int(data.get('Negative', 0))
    )

    words = " ".join(
        df[df['region'] == region]['review']
    ).lower().split()

    common_words = Counter(
        words
    ).most_common(5)

    print("\n🔥 Trending Words:")

    for word, count in common_words:
        print(word, "(", count, ")")

Saving sentiment_regions_500.csv to sentiment_regions_500 (1).csv
✅ Dataset Loaded
                                      review     region sentiment
0  Excellent customer support from Hyderabad  Hyderabad  Positive
1      Amazing shopping experience in Mumbai     Mumbai  Positive
2       Bad purchase experience in Hyderabad  Hyderabad  Negative
3              Terrible service in Hyderabad  Hyderabad  Negative
4     Loved the product quality in Hyderabad  Hyderabad  Positive

✅ Sentiment Analysis Completed

✅ Aggregation Completed
      region     Score  Negative  Neutral  Positive      lat      lon
0  Bangalore  0.166127      30.0      3.0      53.0  12.9716  77.5946
1    Chennai  0.090355      39.0      3.0      56.0  13.0827  80.2707
2      Delhi  0.167207      37.0      5.0      77.0  28.7041  77.1025
3  Hyderabad  0.034288      40.0      2.0      59.0  17.3850  78.4867
4     Mumbai  0.066716      36.0      0.0      60.0  19.0760  72.8777



Available Regions:
['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai']

Enter region name (or type exit): chennai
❌ Region not found

Available Regions:
['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai']

Enter region name (or type exit): Chennai


/tmp/ipykernel_3300/2805093088.py:170: FutureWarning:

Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead

/tmp/ipykernel_3300/2805093088.py:175: FutureWarning:

Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead

/tmp/ipykernel_3300/2805093088.py:180: FutureWarning:

Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead




📍 Region: Chennai
⭐ Avg Score: 0.09
😊 Positive: 56
😐 Neutral: 3
😡 Negative: 39

🔥 Trending Words:
chennai ( 98 )
in ( 90 )
experience ( 31 )
delivery ( 17 )
purchase ( 15 )

Available Regions:
['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai']

Enter region name (or type exit): exit
Program ended.
